In [1]:
# Notebook: nb_gold_risk_scoring
# Purpose: Calculate a 0-100 risk score for each company based on weighted factors

from pyspark.sql import functions as F
from datetime import datetime

print("📊 Starting Gold Layer - Company Risk Scoring...")
print("=" * 60)

# 1. Load Silver and Gold tables
companies_df = spark.table("silver_companies")
sanctions_df = spark.table("gold_sanctions_screening")
jse_df = spark.table("silver_jse_listings")

print(f"🏢 Companies to score: {companies_df.count()}")

# 2. Left join with sanctions to get hit status
scored_df = companies_df.alias("c").join(
    sanctions_df.alias("s"),
    F.col("c.RegistrationNumber") == F.col("s.company_registration"),
    "left"
)

# 3. Left join with JSE to check if listed (using CompanyName for synthetic data match)
scored_df = scored_df.join(
    jse_df.alias("j"),
    F.col("c.CompanyName") == F.col("j.CompanyName"),
    "left"
)

# 4. Calculate Risk Factors (Boolean flags)
scored_df = scored_df.withColumn(
    "is_sanctioned", 
    F.when(F.col("s.sanctions_hit").isNotNull(), True).otherwise(False)
)

scored_df = scored_df.withColumn(
    "is_jse_listed", 
    F.when(F.col("j.JSE_Code").isNotNull(), True).otherwise(False)
)

# Company age in days (using current date)
scored_df = scored_df.withColumn(
    "company_age_days",
    F.datediff(F.current_date(), F.col("c.RegistrationDate"))
).withColumn(
    "is_new_company",
    F.when(F.col("company_age_days") < 365, True).otherwise(False)
)

# Director count risk (1 or fewer directors is higher risk for shell companies)
scored_df = scored_df.withColumn(
    "is_low_directors",
    F.when(F.col("c.DirectorCount") <= 1, True).otherwise(False)
)

# High-risk province (e.g., GAUTENG has higher corporate density and fraud volume)
scored_df = scored_df.withColumn(
    "is_high_risk_province",
    F.when(F.col("c.Province") == "GAUTENG", True).otherwise(False)
)

# 5. Calculate Weighted Risk Score (Base 0, Max 100)
# Sanctions hit = +50 points
# New company (< 1 yr) = +15 points
# Low directors (<= 1) = +20 points
# High-risk province = +10 points
# JSE Listed = -20 points (regulatory scrutiny lowers risk)

scored_df = scored_df.withColumn(
    "raw_risk_score",
    F.when(F.col("is_sanctioned"), 50).otherwise(0) +
    F.when(F.col("is_new_company"), 15).otherwise(0) +
    F.when(F.col("is_low_directors"), 20).otherwise(0) +
    F.when(F.col("is_high_risk_province"), 10).otherwise(0) -
    F.when(F.col("is_jse_listed"), 20).otherwise(0)
)

# Clamp score between 0 and 100
scored_df = scored_df.withColumn(
    "risk_score",
    F.when(F.col("raw_risk_score") < 0, 0)
    .when(F.col("raw_risk_score") > 100, 100)
    .otherwise(F.col("raw_risk_score"))
)

# 6. Assign Risk Category
scored_df = scored_df.withColumn(
    "risk_category",
    F.when(F.col("risk_score") >= 70, "HIGH")
    .when(F.col("risk_score") >= 40, "MEDIUM")
    .otherwise("LOW")
)

# 7. Select final columns for Gold table
final_risk_df = scored_df.select(
    F.col("c.RegistrationNumber").alias("company_registration"),
    F.col("c.CompanyName").alias("company_name"),
    F.col("c.Province").alias("province"),
    F.col("company_age_days").alias("company_age_days"),
    F.col("is_sanctioned").alias("sanctions_hit"),
    F.col("is_jse_listed").alias("jse_listed"),
    F.col("risk_score").cast("int").alias("risk_score"),
    F.col("risk_category").alias("risk_category"),
    F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")).alias("scoring_timestamp"),
    F.lit(100).alias("dq_score")
)

# 8. Write to Gold layer
final_risk_df.write.format("delta").mode("overwrite").saveAsTable("gold_company_risk_scores")

print(f"\n Risk Scoring Complete!")
print(f" Total companies scored: {final_risk_df.count()}")
print("Table created: gold_company_risk_scores")

# 9. Display summary
print("\n Risk Category Distribution:")
display(final_risk_df.groupBy("risk_category").count().orderBy("count", ascending=False))

print("\n Top 5 Highest Risk Companies:")
display(final_risk_df.orderBy(F.col("risk_score").desc()).limit(5))

StatementMeta(, b9fe1eb1-c6a7-481f-aaa4-99b83dd2b7a5, 3, Finished, Available, Finished, False)

📊 Starting Gold Layer - Company Risk Scoring...
🏢 Companies to score: 891

 Risk Scoring Complete!
 Total companies scored: 891
Table created: gold_company_risk_scores

 Risk Category Distribution:


SynapseWidget(Synapse.DataFrame, 1a8c2342-f2c3-44a6-8fab-f59458bab587)


 Top 5 Highest Risk Companies:


SynapseWidget(Synapse.DataFrame, 1e71d770-57e3-4c23-b816-2b4cd4060fca)